# EEG-PRO Zenodo Download & ICA Preprocessing Pipeline
This notebook directly downloads the Zenodo EEG-PRO dataset (Record: 13219018) into the Colab environment.
It also contains the memory-efficient MNE-Python loop to compute ICA on all subjects sequentially.

In [ ]:
# 1. Install dependencies and the Zenodo downloader
!pip install mne mne-icalabel pyriemann zenodo_get

In [ ]:
import os

# 2. Download the Zenodo repository directly to Colab
# This pulls DOI 10.5281/zenodo.13219018
!zenodo_get 13219018

# If the dataset downloads as zip files, uncomment the lines below to extract them:
# !apt-get install unzip
# !unzip '*.zip' -d /content/eeg_data

In [ ]:
# 3. Memory-Efficient ICA Loop
import mne
from mne.preprocessing import ICA
from mne_icalabel import label_components
import glob

# Locate all the downloaded .set files (Adjust this path if you unzipped to a specific folder!)
eeg_files = glob.glob('*.set')  
print(f"Found {len(eeg_files)} subjects.")

# Create an output directory for cleaned files
os.makedirs('clean_eeg_data', exist_ok=True)

for file in eeg_files:
    print(f"\nProcessing {file}...")
    try:
        # A. Load the data (preload=True loads it into RAM)
        raw = mne.io.read_raw_eeglab(file, preload=True)
        
        # B. Filter the data (CRITICAL for ICA: 1 Hz high-pass prevents algorithm failure)
        raw.filter(l_freq=1.0, h_freq=100.0)
        
        # C. Fit ICA
        # We use fastica, preserving 99% of variance
        ica = ICA(n_components=0.99, method='fastica', random_state=42)
        ica.fit(raw)
        
        # D. Auto-Label Components using MNE-ICALabel
        ic_labels = label_components(raw, ica, method='iclabel')
        labels = ic_labels['labels']
        
        # E. Identify Artifacts (Eye blinks and muscle noise)
        exclude_idx = [i for i, label in enumerate(labels) if label in ['eye', 'muscle']]
        print(f"Excluding {len(exclude_idx)} artifact components: {exclude_idx}")
        ica.exclude = exclude_idx
        
        # F. Reconstruct the clean signal by applying the ICA
        raw_clean = ica.apply(raw.copy())
        
        # G. Save the cleaned data (MNE saves as .fif)
        out_name = os.path.join('clean_eeg_data', f"clean_{os.path.basename(file)}")
        raw_clean.save(out_name.replace('.set', '-raw.fif'), overwrite=True)
        
        # H. Clear RAM immediately before the next subject
        del raw, ica, raw_clean
        
    except Exception as e:
        print(f"Failed on {file}: {e}")
        continue

print("\nAll processing complete! Your clean files are in the 'clean_eeg_data' folder.")
# You can zip them up and download them back to your PC:
# !zip -r clean_eeg_data.zip clean_eeg_data/
